# gpubench — Colab Pro validation (real GPU, no Docker)

Colab is already a container, so there is **no Docker here** — we run vLLM
natively. This notebook gets the first *true* numbers on a real GPU before the
full Dockerized AWS run.

**Before you start:** set `Runtime > Change runtime type > GPU` (L4/A100 ideal;
on a 16GB T4 use `--quantization awq` or a smaller model). Add your Hugging Face
token as a Colab **Secret** named `HF_TOKEN`, and make sure that HF account has
**accepted the Llama-3.1 license** on the model page.

In [ ]:
# 1) GPU + install. Always pulls the latest code so re-runs pick up fixes.
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
%cd /content
![ -d llm_inference_benchmarking ] || git clone https://github.com/Saibernard/llm_inference_benchmarking.git
!git -C llm_inference_benchmarking pull --ff-only
%cd /content/llm_inference_benchmarking
!pip -q install vllm==0.11.0
!pip -q install "transformers==4.57.0"  # vLLM 0.11.0 needs transformers<5; Colab ships 5.x
!pip -q install -e .
import sys; sys.path.insert(0, '/content/llm_inference_benchmarking/src')  # importable in THIS kernel

In [ ]:
# 2) HF token from Colab Secrets (never hardcode).
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
# Optional: persist the ~16GB weights to Drive so a session restart doesn't re-download.
# from google.colab import drive; drive.mount('/content/drive')
# os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

In [ ]:
# 3) Launch vLLM natively with the CANONICAL benchmark flags (built from ServeConfig).
import subprocess, time, httpx
from gpubench.config import ServeConfig
from gpubench.serving import build_vllm_serve_cmd
cmd = build_vllm_serve_cmd(ServeConfig(max_model_len=8192))
print(' '.join(cmd))
server = subprocess.Popen(cmd, stdout=open('vllm.log','w'), stderr=subprocess.STDOUT)
for _ in range(120):  # model load can take a few minutes
    try:
        if httpx.get('http://127.0.0.1:8000/health', timeout=2).status_code == 200:
            print('vLLM ready'); break
    except Exception: pass
    time.sleep(5)
else:
    print('NOT ready — check vllm.log')
    print(open('vllm.log').read()[-3000:])

In [ ]:
# 4) Run the open-loop QPS sweep (coordinated-omission-correct) + auto-report.
!gpubench run --config configs/colab.yaml --base-url http://127.0.0.1:8000

In [ ]:
# 5) Cross-check our custom load-gen against vLLM's OFFICIAL oracle on the SAME server.
#    Our numbers should match `vllm bench serve` within tolerance.
!BASE_URL=http://127.0.0.1:8000 MODEL=llama3.1-8b INPUT_LEN=512 OUTPUT_LEN=128 \
  NUM_PROMPTS=200 REQUEST_RATE=16 bash scripts/run_bench_serve_oracle.sh

In [ ]:
# 6) Show the plots + summary from the latest REAL run (skips the empty oracle dir).
import glob, os
from IPython.display import Image, display
import pandas as pd
runs = [d for d in glob.glob('results/*/') if os.path.exists(os.path.join(d, 'summary.csv'))]
latest = max(runs, key=os.path.getmtime)
print('Run:', latest)
display(pd.read_csv(os.path.join(latest, 'summary.csv')))
for p in sorted(glob.glob(os.path.join(latest, 'plots', '*.png'))):
    print(p); display(Image(p))

In [ ]:
# 7) Auto-download ALL results (plots + CSVs) to your machine.
#    Colab wipes /content on disconnect, so this saves a zip to your browser's Downloads.
from google.colab import files
import shutil
shutil.make_archive('gpubench_results', 'zip', 'results')
print('Downloading gpubench_results.zip (plots + summary + raw data) ...')
files.download('gpubench_results.zip')

In [ ]:
# 8) Clean shutdown of the server process.
server.terminate()